# 🚀 CinemaBox Cloud AI Server (5 TB Google Storage)
### Powered by Qwen 2.5 32B Instruct (128k Context | SOTA Intelligence)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_70b_google_storage.ipynb)

---

## ⚡ 1-Click Launch (Clean & Fresh Setup)
Press the **Play** button below. This script will:
1. 🧹 **Clean local Colab disk** to ensure zero "disk full" errors.
2. 📁 **Mount your 5 TB Google Drive** (`/content/drive/MyDrive/CinemaBox_Models/`).
3. ⚡ **Install GPU acceleration & Cloudflare tunnel** in ~30 seconds.
4. 📦 **Store the all-in-one powerhouse model** (`Qwen2.5-32B-Instruct`, 128k context, beats 70B models).
5. 🌐 **Launch your public HTTPS URL** ready to use in **CinemaBox**, **VS Code**, and browser.

In [ ]:
#@title 🚀 Click Play to Start AI Server { vertical-output: true }
import os
import sys
import time
import subprocess
import re

print("="*70)
print("🚀 STARTING CINEMABOX ALL-IN-ONE AI SERVER...")
print("="*70)

# -----------------------------------------------------------------------
# 1. Clean Local Disk to Prevent Any Space Errors
# -----------------------------------------------------------------------
print("\n[1/5] 🧹 Cleaning local VM disk...")
!rm -rf /root/.cache/huggingface /content/sample_data
print("      ✅ Local disk cleaned successfully!")

# -----------------------------------------------------------------------
# 2. Mount 5 TB Google Drive
# -----------------------------------------------------------------------
print("\n[2/5] 📁 Mounting your 5 TB Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['HF_HOME'] = os.path.join(DRIVE_DIR, '.cache')
print(f"      ✅ Connected to 5 TB Google Drive: {DRIVE_DIR}")

# -----------------------------------------------------------------------
# 3. Install High-Speed GPU Engine & Cloudflare Tunnel
# -----------------------------------------------------------------------
print("\n[3/5] ⚡ Installing GPU Engine & Cloudflare Tunnel...")
!pip install --upgrade pip -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
!pip install "llama-cpp-python[server]" fastapi uvicorn huggingface_hub requests -q
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("      ✅ GPU Engine & Cloudflare Tunnel ready!")

# -----------------------------------------------------------------------
# 4. Download & Cache the Best All-in-One Model (Qwen 2.5 32B Instruct)
# -----------------------------------------------------------------------
print("\n[4/5] 📦 Loading Qwen 2.5 32B (SOTA Intelligence, 128k context, Coding + Cinema)...")
from huggingface_hub import hf_hub_download

REPO_ID = "bartowski/Qwen2.5-32B-Instruct-GGUF"
FILENAME = "Qwen2.5-32B-Instruct-Q4_K_M.gguf"
model_path = os.path.join(DRIVE_DIR, FILENAME)

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"      🎯 Reusing permanently cached model in 5 TB Drive: {FILENAME}")
    print(f"         Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print(f"      📥 Downloading {FILENAME} directly into your 5 TB Google Drive...")
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        local_dir=DRIVE_DIR,
        local_dir_use_symlinks=False,
        cache_dir=os.path.join(DRIVE_DIR, ".cache")
    )
    print(f"      ✅ Download complete! Saved permanently in: {model_path}")

# -----------------------------------------------------------------------
# 5. Launch OpenAI-Compatible Server & Cloudflare Tunnel
# -----------------------------------------------------------------------
print("\n[5/5] 🌐 Starting OpenAI-Compatible Server & Cloudflare Public HTTPS Tunnel...")
API_KEY = "sk-cinemabox70b"

server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",
    "--n_ctx", "8192",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api_key", API_KEY,
    "--chat_format", "chatml"
]

server_proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10) # Initialize neural weights in GPU VRAM

# Start Cloudflare tunnel
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print("🎉 YOUR AI SERVER IS LIVE GLOBALLY AND READY TO USE!")
    print("="*70)
    print(f"\n👉 📱 CinemaBox App Endpoint:     {public_url}/v1/chat/completions")
    print(f"👉 💻 VS Code (Roo/Continue) Base: {public_url}/v1")
    print(f"👉 🔑 API Key:                     {API_KEY}")
    print(f"👉 🧠 Active Model:                Qwen2.5-32B-Instruct")
    print("\n" + "-"*70)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print(f"   1. Open CinemaBox on your phone > Settings > On-Device Intelligence")
    print(f"   2. Tap the '141B MoE / VISION' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print(f"   4. Tap Save & Run Test -> You are connected!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print(f"   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  {API_KEY}")
    print(f"   Model:    Qwen2.5-32B-Instruct")
    print("="*70 + "\n")
else:
    print("❌ Cloudflare tunnel could not be resolved. Please re-run the cell.")

--- 
### Step 2 (Optional): Test Inference Live
Run this cell to test your live model directly from Colab.

In [ ]:
import requests

prompt = "Recommend 3 mind-bending sci-fi mystery movies for a viewer who loves Interstellar and Arrival. Include a 1-sentence hook for each."

print(f"💬 Sending prompt: '{prompt}'\n")
res = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Authorization": "Bearer sk-cinemabox70b"},
    json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 350,
        "temperature": 0.7
    }
)

if res.status_code == 200:
    print("🎬 Output:\n")
    print(res.json()["choices"][0]["message"]["content"])
else:
    print(f"Error {res.status_code}: {res.text}")